# Model 4 — Parallel Preload + Experiment Tracking
**UCSF-PDGM Brain Tumor Classification | NIfTI MRI | ResNet50**

This notebook extends `Model 2_parallel_preload.ipynb` with a full **experiment tracking system**
designed for iterative hyperparameter search.

## Workflow
1. **Run cells 1–22 once per session** (scan patients, init RunManager — ~1 min, mostly I/O)
2. **Edit cells 24–26 for each new run** (change `RunCfg`, `TrainCfg`, optionally `PreprocCfg`)
3. **Run cells 24–27** — builds dataset, trains, evaluates, saves everything automatically
4. **Open `reports_viz.ipynb`** — pick a run from the dropdown, compare results

## What gets saved per run
| File | Location | Contents |
|------|----------|----------|
| `runs.csv` | `Reports/` | One row: all configs + best val metrics + timing |
| `epochs.csv` | `Reports/` | One row per epoch: loss, all val metrics, LR |
| `test_reports.csv` | `Reports/` | One row: test metrics |
| `{run_id}.pt` | `Reports/best_models/` | Best model weights |
| `{run_id}.npz` | `Reports/pr_curves/` | Full precision-recall curve arrays |
| `column_reference.md` | `Reports/` | Auto-generated column documentation |

## Tips for effective iteration
- **Always fill `RunCfg.note` and `RunCfg.tag`** — they're the 'commit message' for your experiment.
  Good notes make the config diff table in the viz notebook instantly interpretable.
- **Use `RunCfg.seed`** to seed everything before training. Two runs with identical configs
  but different seeds can give different results — seed explicitly when you want to isolate the
  effect of a single hyperparameter change.
- **Primary metric is `val_ap` (PR-AUC)**, not val_f1 or balanced_acc. AP is threshold-free and
  more robust to class imbalance. The threshold is tuned on the val set each epoch — treat
  it as a downstream decision parameter, not a model selection criterion.
- **Data fingerprint**: `runs.csv` stores an MD5 hash of your patient list. If two runs have
  different fingerprints their metrics are NOT comparable (dataset changed).
- **Optional future extension**: save a checkpoint every K epochs to `Reports/checkpoints/`
  so you can resume interrupted runs or plot model evolution over training.

In [1]:
import os
import re
import csv
import sys
import math
import time
import random
import hashlib
import traceback
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Literal

import numpy as np
import pandas as pd
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    f1_score, confusion_matrix, balanced_accuracy_score, roc_auc_score,
)

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import warnings
warnings.filterwarnings('ignore')

print(f"[Init] Python {sys.version}")
print(f"[Init] PyTorch {torch.__version__}")
print(f"[Init] CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" — {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print()
print(f"[Init] CPU cores: {os.cpu_count()}")

[Init] Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
[Init] PyTorch 2.10.0+cu128
[Init] CUDA: True — NVIDIA GeForce RTX 5070 Ti (17.1 GB)
[Init] CPU cores: 16


## Configuration

- **`PreprocCfg`** — preprocessing parameters (modalities, resolution, normalization, slice selection)
- **`TrainCfg`** — training hyperparameters (lr, epochs, loss type, scheduler)
- **`ParallelCfg`** — parallel data loader settings
- **`RunCfg`** *(new)* — run metadata: human-readable note, short tag, and random seed

In [2]:
@dataclass
class PreprocCfg:
    mods: Tuple[str, ...] = ("T1", "T1c", "T2", "FLAIR")
    out_hw: Tuple[int, int] = (256, 256)
    topk: int = 24
    context_slices: int = 2
    include_context_ratio: float = 0.25
    bbox_margin: int = 24
    min_bbox_size: int = 32
    norm_mode: Literal["robust_z", "pct01"] = "robust_z"
    clip_z: float = 5.0
    eps: float = 1e-6
    add_tumor_mask_channel: bool = False


@dataclass
class TrainCfg:
    epochs: int = 15
    lr: float = 3e-4
    weight_decay: float = 1e-4
    amp: bool = True
    grad_clip: float = 1.0
    loss_type: Literal['weighted_ce', 'focal'] = "weighted_ce"
    focal_gamma: float = 2.0
    max_lgg_weight: float = 8.0
    use_cosine: bool = True
    warmup_epochs: int = 1
    early_stop_patience: int = 5
    tune_threshold_on: Literal["f1", "recall>=X"] = "f1"
    min_recall: float = 0.70
    patient_agg: Literal['mean', "max"] = "mean"


@dataclass
class ParallelCfg:
    """Controls the parallel dataset builder."""
    n_workers: int = -1
    device: Literal['cuda', 'cpu'] = "cuda" if torch.cuda.is_available() else "cpu"
    max_ram_estimate_gb: float = 12.0
    verbose: bool = True

    def resolved_n_workers(self) -> int:
        if self.n_workers <= 0:
            return max(1, (os.cpu_count() or 2) // 2)
        return self.n_workers


@dataclass
class RunCfg:
    """
    Metadata attached to each training run.
    Fill these EVERY run — they are the 'commit message' for your experiment.
    """
    note: str = ""     # free-text: what is different about this run?
    tag:  str = ""     # short suffix for run_id, e.g. 'focal', 'lr1e4', 'topk32'
    seed: int = 42     # seeds torch + numpy + random for reproducibility


_dev = ParallelCfg().device
print(f"[Config] Default device  : {_dev}")
print(f"[Config] Default workers : {ParallelCfg().resolved_n_workers()} of {os.cpu_count()} cores")
print(f"[Config] PreprocCfg defaults: topk={PreprocCfg().topk}, ctx=±{PreprocCfg().context_slices}, "
      f"out={PreprocCfg().out_hw}, norm={PreprocCfg().norm_mode}")

[Config] Default device  : cuda
[Config] Default workers : 8 of 16 cores
[Config] PreprocCfg defaults: topk=24, ctx=±2, out=(256, 256), norm=robust_z


## Core Preprocessing Utilities
Verbatim from `Model 2_parallel_preload.ipynb`. Kept at top-level for picklability.

In [3]:
def load_nii(path: str) -> np.ndarray:
    return nib.load(path).get_fdata().astype(np.float32)

def safe_percentile(x: np.ndarray, q: float) -> float:
    return float(np.percentile(x, q)) if x.size > 0 else 0.0

def clip01(x: np.ndarray) -> np.ndarray:
    return np.clip(x, 0.0, 1.0)

def normalize_volume(vol: np.ndarray, parenchyma_mask: np.ndarray, cfg: PreprocCfg) -> np.ndarray:
    m = parenchyma_mask > 0
    v = vol[m]
    if v.size == 0: v = vol[vol != 0]
    if v.size == 0: return vol
    if cfg.norm_mode == "robust_z":
        med = float(np.median(v))
        iqr = safe_percentile(v, 75) - safe_percentile(v, 25)
        scale = iqr if iqr > cfg.eps else float(np.std(v) + cfg.eps)
        return np.clip((vol - med) / (scale + cfg.eps), -cfg.clip_z, cfg.clip_z).astype(np.float32)
    elif cfg.norm_mode == "pct01":
        p1, p99 = safe_percentile(v, 1), safe_percentile(v, 99)
        denom = (p99 - p1) if (p99 - p1) > cfg.eps else 1.0
        return clip01((vol - p1) / denom).astype(np.float32)
    raise ValueError(f"Unknown norm_mode: {cfg.norm_mode}")

def tumor_area_per_slice(tumor_mask: np.ndarray, brain_mask: np.ndarray) -> np.ndarray:
    tumor, brain = tumor_mask > 0, brain_mask > 0
    D = tumor_mask.shape[2]
    frac = np.zeros((D,), dtype=np.float32)
    for z in range(D):
        denom = float(np.count_nonzero(brain[..., z]))
        frac[z] = float(np.count_nonzero(tumor[..., z])) / denom if denom > 0 else 0.0
    return frac

def select_slices_topk_with_context(tumor_mask: np.ndarray, brain_mask: np.ndarray,
                                     cfg: PreprocCfg) -> List[int]:
    frac = tumor_area_per_slice(tumor_mask, brain_mask)
    D = frac.shape[0]
    if np.all(frac == 0):
        mid = D // 2
        return list(range(max(0, mid - cfg.topk // 2), min(D, mid + cfg.topk // 2)))
    top_idx = np.argsort(frac)[-min(cfg.topk, D):].tolist()
    ctx = set(top_idx)
    for z in top_idx:
        for dz in range(1, cfg.context_slices + 1):
            ctx.add(max(0, z - dz)); ctx.add(min(D - 1, z + dz))
    return sorted(ctx)

def bbox_from_mask2d(mask2d: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    ys, xs = np.where(mask2d > 0)
    if ys.size == 0: return None
    return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1

def expand_bbox(bbox, H, W, margin, min_size):
    r0, r1, c0, c1 = bbox
    r0 = max(0, r0 - margin); r1 = min(H, r1 + margin)
    c0 = max(0, c0 - margin); c1 = min(W, c1 + margin)
    if r1 - r0 < min_size:
        ctr = (r0 + r1) // 2; r0 = max(0, ctr - min_size//2); r1 = min(H, r0 + min_size)
    if c1 - c0 < min_size:
        ctr = (c0 + c1) // 2; c0 = max(0, ctr - min_size//2); c1 = min(W, c0 + min_size)
    return r0, r1, c0, c1

def crop_and_resize_multich(img_ch: np.ndarray, crop_bbox, out_hw) -> np.ndarray:
    r0, r1, c0, c1 = crop_bbox
    crop = img_ch[:, r0:r1, c0:c1]
    t = torch.from_numpy(crop).unsqueeze(0)
    t = F.interpolate(t, size=out_hw, mode="bilinear", align_corners=False)
    return t.squeeze(0).numpy().astype(np.float32)

print("[OK] Preprocessing utilities defined.")

[OK] Preprocessing utilities defined.


## Patient Metadata & File Scanning
Verbatim from `Model 2_parallel_preload.ipynb`.

In [4]:
@dataclass
class PatientRecord:
    pid: str; label: int
    mods: Dict[str, str]; masks: Dict[str, str]; folder: str

def _pick_modality_file(files, pid, mod, prefer_bias):
    base = f"{pid}_{mod}.nii.gz"; bias = f"{pid}_{mod}_bias.nii.gz"
    if prefer_bias and bias in files: return bias
    return base if base in files else None

def _pick_mask_file(files, pid, kind):
    m = {"brain": f"{pid}_brain_segmentation.nii.gz",
         "parenchyma": f"{pid}_brain_parenchyma_segmentation.nii.gz",
         "tumor": f"{pid}_tumor_segmentation.nii.gz"}
    return m[kind] if m[kind] in files else None

def normalize_ucsf_pid(pid):
    if pid is None: return None
    m = re.match(r"^(UCSF-PDGM-)(\d+)$", str(pid).strip())
    return f"{m.group(1)}{int(m.group(2)):04d}" if m else None

def build_labels_from_pdgm_v5(csv_path, id_col="ID", grade_col="WHO CNS Grade",
                                low_grades=(1,2), high_grades=(3,4)):
    df = pd.read_csv(csv_path)
    df["pid_norm"]  = df[id_col].apply(normalize_ucsf_pid)
    df["grade_num"] = pd.to_numeric(df[grade_col], errors="coerce")
    df = df.dropna(subset=["pid_norm", "grade_num"]).copy()
    df["grade_num"] = df["grade_num"].astype(int)
    def g2l(g): return 1 if g in low_grades else (0 if g in high_grades else None)
    df["label"] = df["grade_num"].apply(g2l)
    df = df.dropna(subset=["label"]).copy(); df["label"] = df["label"].astype(int)
    labels = dict(zip(df["pid_norm"], df["label"]))
    print(f"[Labels] {len(labels)} | LGG(1)={(df['label']==1).sum()} | HGG(0)={(df['label']==0).sum()}")
    return labels, df

def scan_ucsf_pdgm_v5(root_dir, labels_by_pid, prefer_bias=True,
                       require_all_modalities=True, require_all_masks=True,
                       mods=("T1","T1c","T2","FLAIR")):
    pat_re = re.compile(r"^(UCSF-PDGM-\d{4})_nifti$")
    patients, skipped = [], []
    print(f"[Scan] {root_dir}")
    for name in sorted(os.listdir(root_dir)):
        m = pat_re.match(name)
        if not m: continue
        pid = m.group(1); folder = os.path.join(root_dir, name)
        if pid not in labels_by_pid: skipped.append((pid, "no label")); continue
        files = set(f for f in os.listdir(folder) if f.endswith(".nii.gz"))
        mod_paths = {}
        miss_m = [mod for mod in mods if not (f := _pick_modality_file(list(files), pid, mod, prefer_bias)) or mod_paths.update({mod: os.path.join(folder, f)}) or False]
        mask_paths = {}
        miss_k = [k for k in ("brain","parenchyma","tumor") if not (f := _pick_mask_file(list(files), pid, k)) or mask_paths.update({k: os.path.join(folder, f)}) or False]
        if require_all_modalities and miss_m: skipped.append((pid, f"miss mods:{miss_m}")); continue
        if require_all_masks and miss_k: skipped.append((pid, f"miss masks:{miss_k}")); continue
        patients.append(PatientRecord(pid=pid, label=int(labels_by_pid[pid]),
                                      mods=mod_paths, masks=mask_paths, folder=folder))
    print(f"[Scan] Found {len(patients)} | Skipped {len(skipped)}")
    if skipped[:3]: print(f"[Scan] First skipped: {skipped[:3]}")
    return patients

def stratified_patient_split(patients, test_size=0.15, val_size=0.15, seed=42):
    y = np.array([p.label for p in patients]); idx = np.arange(len(patients))
    idx_tv, idx_test = train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)
    idx_tr, idx_val  = train_test_split(idx_tv, test_size=val_size/(1-test_size),
                                         random_state=seed, stratify=y[idx_tv])
    def _c(ps): yy = np.array([p.label for p in ps]); return {"N":len(ps),"LGG":(yy==1).sum(),"HGG":(yy==0).sum()}
    tr, va, te = [patients[i] for i in idx_tr], [patients[i] for i in idx_val], [patients[i] for i in idx_test]
    print(f"[Split] Train:{_c(tr)} Val:{_c(va)} Test:{_c(te)}")
    return tr, va, te

print("[OK] Patient utilities defined.")

[OK] Patient utilities defined.


## Parallel Worker: `process_one_patient`
Top-level function — opens each NIfTI file once and returns all selected slices for that patient.

In [5]:
def process_one_patient(pr: PatientRecord, cfg: PreprocCfg):
    try:
        brain = load_nii(pr.masks["brain"]) > 0
        tumor = load_nii(pr.masks["tumor"]) > 0
        paren = load_nii(pr.masks["parenchyma"]) > 0
        vols  = {}
        for mod in cfg.mods:
            v = load_nii(pr.mods[mod])
            vols[mod] = normalize_volume(v, paren.astype(np.uint8), cfg)
        selected_zs = select_slices_topk_with_context(tumor, brain, cfg)
        H, W = brain.shape[0], brain.shape[1]
        results = []
        for z in selected_zs:
            slice_ch = np.stack([vols[m][..., z] for m in cfg.mods], axis=0).astype(np.float32)
            t2d, b2d = tumor[..., z].astype(np.uint8), brain[..., z].astype(np.uint8)
            bbox = bbox_from_mask2d(t2d) or bbox_from_mask2d(b2d)
            if bbox is None:
                bbox = (max(0,H//2-64), min(H,H//2+64), max(0,W//2-64), min(W,W//2+64))
            bbox = expand_bbox(bbox, H, W, cfg.bbox_margin, cfg.min_bbox_size)
            x = crop_and_resize_multich(slice_ch, bbox, cfg.out_hw)
            if cfg.add_tumor_mask_channel:
                tm = crop_and_resize_multich(t2d[None,...].astype(np.float32), bbox, cfg.out_hw)
                x  = np.concatenate([x, tm], axis=0)
            results.append((x.astype(np.float32), int(pr.label), pr.pid, int(z)))
        return results
    except Exception as exc:
        print(f"\n[WARN] process_one_patient({pr.pid}): {exc}")
        traceback.print_exc()
        return []

print("[OK] process_one_patient defined.")

[OK] process_one_patient defined.


## `PreloadedSliceDataset` and `PreloadedPatientBalancedSampler`
`__getitem__` is a pure `O(1)` tensor index — zero preprocessing during training.

In [6]:
class PreloadedSliceDataset(Dataset):
    def __init__(self, X, y, pids, zs):
        assert X.shape[0] == y.shape[0] == len(pids) == len(zs)
        self.X = X; self.y = y; self.pids = pids; self.zs = zs
    def __len__(self):  return self.X.shape[0]
    def __getitem__(self, idx): return self.X[idx], self.y[idx], self.pids[idx], self.zs[idx]
    @property
    def unique_pids(self):
        seen, s = [], set()
        for p in self.pids:
            if p not in s: seen.append(p); s.add(p)
        return seen
    def class_counts(self):
        y = self.y.cpu().numpy()
        return {int(c): int((y==c).sum()) for c in np.unique(y)}
    def summary(self):
        return (f"PreloadedSliceDataset | {len(self)} slices | "
                f"{len(self.unique_pids)} patients | X={tuple(self.X.shape)} "
                f"{self.X.dtype} on {self.X.device} | {self.class_counts()}")


class PreloadedPatientBalancedSampler(Sampler):
    def __init__(self, dataset: PreloadedSliceDataset, batch_patients_per_class=4, seed=42):
        self.ds = dataset; self.k = batch_patients_per_class
        self.rng = np.random.default_rng(seed)
        pid_items: Dict[str, List[int]] = {}
        pid_label: Dict[str, int] = {}
        for i in range(len(dataset)):
            p = dataset.pids[i]; lbl = int(dataset.y[i].item())
            pid_items.setdefault(p, []).append(i); pid_label[p] = lbl
        self.pid_items = pid_items
        self.cls_pids: Dict[int, List[str]] = {0: [], 1: []}
        for p, l in pid_label.items(): self.cls_pids[l].append(p)
        self.num_batches = max(1, len(self.cls_pids[0]) // self.k)
        print(f"[Sampler] HGG={len(self.cls_pids[0])} LGG={len(self.cls_pids[1])} "
              f"k={self.k} ~{self.num_batches} batches/epoch")
    def __len__(self): return self.num_batches
    def __iter__(self):
        p0 = self.cls_pids[0].copy(); p1 = self.cls_pids[1].copy()
        self.rng.shuffle(p0); self.rng.shuffle(p1)
        def _cy(lst):
            while True:
                for x in lst: yield x
        it0, it1 = iter(p0), _cy(p1)
        for _ in range(self.num_batches):
            s0 = [next(it0, None) for _ in range(self.k)]; s0 = [x for x in s0 if x]
            if len(s0) < self.k:
                self.rng.shuffle(p0); it0 = iter(p0)
                s0 += [next(it0) for _ in range(self.k - len(s0))]
            s1 = [next(it1) for _ in range(self.k)]
            yield [int(self.rng.choice(self.pid_items[p])) for p in s0 + s1]

print("[OK] PreloadedSliceDataset and PreloadedPatientBalancedSampler defined.")

[OK] PreloadedSliceDataset and PreloadedPatientBalancedSampler defined.


## `ParallelDatasetBuilder`
Three-phase pipeline: estimate workload → parallel patient processing → stack into tensor bank.

In [7]:
class ParallelDatasetBuilder:
    @staticmethod
    def _est_bytes(n, cfg): return n * (len(cfg.mods)+(1 if cfg.add_tumor_mask_channel else 0)) * cfg.out_hw[0] * cfg.out_hw[1] * 4

    @staticmethod
    def build(patients, cfg, parallel_cfg, split_name="dataset"):
        t0 = time.perf_counter(); SEP = "="*65
        n = len(patients)
        est_s  = n * (cfg.topk + 2*cfg.context_slices)
        est_gb = ParallelDatasetBuilder._est_bytes(est_s, cfg) / 1e9
        print(f"\n{SEP}\n[Phase 1/3 | {split_name}] Estimate: {n} patients, ~{est_s} slices, ~{est_gb:.2f} GB")
        if est_gb > parallel_cfg.max_ram_estimate_gb:
            print(f"  [WARN] Exceeds max_ram_estimate_gb={parallel_cfg.max_ram_estimate_gb}. Consider reducing topk or using device='cpu'.")

        nw  = parallel_cfg.resolved_n_workers()
        print(f"[Phase 2/3 | {split_name}] Processing ({nw} threads) ...")
        all_slices, failed = [], []
        p2t0 = time.perf_counter()

        with ThreadPoolExecutor(max_workers=nw) as ex:
            fut2pid = {ex.submit(process_one_patient, pr, cfg): pr.pid for pr in patients}
            with tqdm(total=n, desc=f"  [{split_name}]", unit="pt") as pb:
                for ft in as_completed(fut2pid):
                    pid = fut2pid[ft]
                    try:
                        sl = ft.result(); all_slices.extend(sl)
                        pb.set_postfix(slices=len(all_slices), pid=pid[-8:], fail=len(failed))
                    except Exception as e:
                        failed.append(pid); print(f"  [WARN] {pid}: {e}")
                    pb.update(1)

                p2e = time.perf_counter() - p2t0
        print(f"  Done: {n-len(failed)}/{n} OK | {len(failed)} failed | {len(all_slices)} slices | {p2e:.1f}s")
        if failed: print(f"  Failed: {failed}")
        if not all_slices: raise RuntimeError(f"[{split_name}] No slices collected.")

        print(f"[Phase 3/3 | {split_name}] Stacking {len(all_slices)} slices ...")
        p3t0 = time.perf_counter()
        all_slices.sort(key=lambda t: (t[2], t[3]))
        xs = np.stack([s[0] for s in all_slices], axis=0)
        ys_i = [s[1] for s in all_slices]; pids = [s[2] for s in all_slices]; zs = [s[3] for s in all_slices]
        print(f"  Shape: {xs.shape} | {xs.nbytes/1e9:.2f} GB")
        X = torch.from_numpy(xs).contiguous().to(parallel_cfg.device, non_blocking=True)
        y = torch.tensor(ys_i, dtype=torch.long).to(parallel_cfg.device, non_blocking=True)
        del all_slices, xs
        p3e = time.perf_counter() - p3t0
        total = time.perf_counter() - t0
        ds = PreloadedSliceDataset(X, y, pids, zs)
        print(f"  Stack+transfer: {p3e:.1f}s | Total: {total:.1f}s\n  {ds.summary()}\n{SEP}")
        return ds

print("[OK] ParallelDatasetBuilder defined.")

[OK] ParallelDatasetBuilder defined.


## Run Tracking System: `RunManager`

Manages all persistent run artifacts: CSV files, model files, PR curve files, and documentation.
Auto-creates `Reports/` directories and CSV headers on first use.
Also generates `column_reference.md` — a human-readable description of every column.

In [8]:
class RunManager:
    """Manages all persistent artifacts for experiment runs."""

    RUNS_COLS = [
        "run_id", "run_note", "tag", "seed",
        "timestamp_start", "timestamp_end", "duration_s",
        "n_train_patients", "n_val_patients", "n_test_patients",
        "n_train_slices", "n_val_slices", "n_test_slices",
        "hgg_train", "lgg_train",
        "epochs_planned", "epochs_actual", "best_epoch",
        "best_val_ap", "best_val_f1", "best_val_balanced_acc",
        "best_val_precision", "best_val_recall", "best_val_threshold",
        "best_model_path",
        "topk", "context_slices", "out_hw", "norm_mode",
        "add_tumor_mask_channel", "bbox_margin", "min_bbox_size",
        "epochs_cfg", "lr", "weight_decay", "loss_type", "amp",
        "grad_clip", "focal_gamma", "max_lgg_weight",
        "use_cosine", "warmup_epochs", "early_stop_patience", "patient_agg",
        "device", "gpu_name", "n_workers", "data_fingerprint",
    ]
    EPOCHS_COLS = [
        "run_id", "epoch", "train_loss", "epoch_time_s", "lr_current", "is_best",
        "val_ap", "val_roc_auc", "val_f1", "val_balanced_acc",
        "val_precision", "val_recall", "val_threshold",
        "val_tn", "val_fp", "val_fn", "val_tp",
    ]
    TEST_COLS = [
        "run_id", "timestamp", "n_test_patients",
        "test_ap", "test_roc_auc", "test_f1", "test_balanced_acc",
        "test_precision", "test_recall", "test_threshold",
        "test_tn", "test_fp", "test_fn", "test_tp",
    ]

    def __init__(self, reports_dir):
        self.reports_dir = Path(reports_dir)
        self.models_dir  = self.reports_dir / "best_models"
        self.pr_dir      = self.reports_dir / "pr_curves"
        self.reports_dir.mkdir(parents=True, exist_ok=True)
        self.models_dir.mkdir(exist_ok=True)
        self.pr_dir.mkdir(exist_ok=True)
        self.runs_csv   = self.reports_dir / "runs.csv"
        self.epochs_csv = self.reports_dir / "epochs.csv"
        self.test_csv   = self.reports_dir / "test_reports.csv"
        for path, cols in [(self.runs_csv, self.RUNS_COLS),
                           (self.epochs_csv, self.EPOCHS_COLS),
                           (self.test_csv, self.TEST_COLS)]:
            if not path.exists():
                with open(path, "w", newline="") as f:
                    csv.DictWriter(f, fieldnames=cols).writeheader()
                print(f"[RunManager] Created {path.name}")
        print(f"[RunManager] Ready → {self.reports_dir}")

    def _clean_tag(self, tag: str) -> str:
        tag = re.sub(r"[^a-z0-9\-]", "-", tag.strip().lower())
        return re.sub(r"-+", "-", tag).strip("-")[:32]

    def new_run_id(self, tag: str = "") -> str:
        base = datetime.now().strftime("run_%Y%m%d_%H%M%S")
        clean = self._clean_tag(tag)
        run_id = f"{base}_{clean}" if clean else base
        # Collision guard (same-second)
        if self.runs_csv.exists():
            try:
                existing = pd.read_csv(self.runs_csv)["run_id"].tolist()
                if run_id in existing:
                    import random as _r; run_id = f"{run_id}_{_r.randint(0,255):02x}"
            except Exception:
                pass
        return run_id

    def _fingerprint(self, patients) -> str:
        return hashlib.md5("|".join(sorted(p.pid for p in patients)).encode()).hexdigest()[:12]

    def start_run(self, run_id, run_cfg, preproc_cfg, train_cfg,
                  split_counts, slice_counts, class_counts, n_workers, device,
                  all_patients=None):
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
        fp  = self._fingerprint(all_patients) if all_patients else ""
        row = {
            "run_id": run_id, "run_note": run_cfg.note, "tag": run_cfg.tag, "seed": run_cfg.seed,
            "timestamp_start": datetime.now().isoformat(), "timestamp_end": "", "duration_s": "",
            "n_train_patients": split_counts.get("train",""), "n_val_patients": split_counts.get("val",""),
            "n_test_patients": split_counts.get("test",""),
            "n_train_slices": slice_counts.get("train",""), "n_val_slices": slice_counts.get("val",""),
            "n_test_slices": slice_counts.get("test",""),
            "hgg_train": class_counts.get("hgg",""), "lgg_train": class_counts.get("lgg",""),
            "epochs_planned": train_cfg.epochs, "epochs_actual": "", "best_epoch": "",
            "best_val_ap": "", "best_val_f1": "", "best_val_balanced_acc": "",
            "best_val_precision": "", "best_val_recall": "", "best_val_threshold": "",
            "best_model_path": "",
            "topk": preproc_cfg.topk, "context_slices": preproc_cfg.context_slices,
            "out_hw": str(preproc_cfg.out_hw), "norm_mode": preproc_cfg.norm_mode,
            "add_tumor_mask_channel": preproc_cfg.add_tumor_mask_channel,
            "bbox_margin": preproc_cfg.bbox_margin, "min_bbox_size": preproc_cfg.min_bbox_size,
            "epochs_cfg": train_cfg.epochs, "lr": train_cfg.lr, "weight_decay": train_cfg.weight_decay,
            "loss_type": train_cfg.loss_type, "amp": train_cfg.amp, "grad_clip": train_cfg.grad_clip,
            "focal_gamma": train_cfg.focal_gamma, "max_lgg_weight": train_cfg.max_lgg_weight,
            "use_cosine": train_cfg.use_cosine, "warmup_epochs": train_cfg.warmup_epochs,
            "early_stop_patience": train_cfg.early_stop_patience, "patient_agg": train_cfg.patient_agg,
            "device": device, "gpu_name": gpu, "n_workers": n_workers, "data_fingerprint": fp,
        }
        with open(self.runs_csv, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.RUNS_COLS).writerow(row)
        print(f"[RunManager] Run started: {run_id}")

    def log_epoch(self, run_id, epoch, train_loss, epoch_time_s, lr, val_report, is_best):
        row = {
            "run_id": run_id, "epoch": epoch,
            "train_loss": round(float(train_loss), 6),
            "epoch_time_s": round(float(epoch_time_s), 2),
            "lr_current": float(lr), "is_best": int(is_best),
            "val_ap":           val_report.get("ap_pr_auc(LGG)", ""),
            "val_roc_auc":      val_report.get("roc_auc", ""),
            "val_f1":           val_report.get("f1(LGG)", ""),
            "val_balanced_acc": val_report.get("balanced_acc", ""),
            "val_precision":    val_report.get("precision(LGG)", ""),
            "val_recall":       val_report.get("recall(LGG)", ""),
            "val_threshold":    val_report.get("threshold", ""),
            "val_tn": val_report.get("tn",""), "val_fp": val_report.get("fp",""),
            "val_fn": val_report.get("fn",""), "val_tp": val_report.get("tp",""),
        }
        with open(self.epochs_csv, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.EPOCHS_COLS).writerow(row)

    def finish_run(self, run_id, epochs_actual, best_epoch, best_metrics, model_path, duration_s):
        df = pd.read_csv(self.runs_csv); mask = df["run_id"] == run_id
        if not mask.any(): print(f"[RunManager][WARN] {run_id} not in runs.csv"); return
        df.loc[mask, "timestamp_end"]       = datetime.now().isoformat()
        df.loc[mask, "duration_s"]          = round(float(duration_s), 2)
        df.loc[mask, "epochs_actual"]       = int(epochs_actual)
        df.loc[mask, "best_epoch"]          = int(best_epoch)
        df.loc[mask, "best_val_ap"]         = best_metrics.get("ap_pr_auc(LGG)", "")
        df.loc[mask, "best_val_f1"]         = best_metrics.get("f1(LGG)", "")
        df.loc[mask, "best_val_balanced_acc"] = best_metrics.get("balanced_acc", "")
        df.loc[mask, "best_val_precision"]  = best_metrics.get("precision(LGG)", "")
        df.loc[mask, "best_val_recall"]     = best_metrics.get("recall(LGG)", "")
        df.loc[mask, "best_val_threshold"]  = best_metrics.get("threshold", "")
        df.loc[mask, "best_model_path"]     = str(model_path)
        df.to_csv(self.runs_csv, index=False)
        ap = best_metrics.get('ap_pr_auc(LGG)', 'N/A')
        ap_str = f"{float(ap):.4f}" if ap != 'N/A' else 'N/A'
        print(f"[RunManager] Run finished: {run_id} | {duration_s:.0f}s | best_val_ap={ap_str}")

    def log_test(self, run_id, test_report):
        row = {
            "run_id": run_id, "timestamp": datetime.now().isoformat(),
            "n_test_patients": test_report.get("n_patients", ""),
            "test_ap":           test_report.get("ap_pr_auc(LGG)", ""),
            "test_roc_auc":      test_report.get("roc_auc", ""),
            "test_f1":           test_report.get("f1(LGG)", ""),
            "test_balanced_acc": test_report.get("balanced_acc", ""),
            "test_precision":    test_report.get("precision(LGG)", ""),
            "test_recall":       test_report.get("recall(LGG)", ""),
            "test_threshold":    test_report.get("threshold", ""),
            "test_tn": test_report.get("tn",""), "test_fp": test_report.get("fp",""),
            "test_fn": test_report.get("fn",""), "test_tp": test_report.get("tp",""),
        }
        with open(self.test_csv, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.TEST_COLS).writerow(row)
        print(f"[RunManager] Test logged for {run_id}")

    def log_pr_curve(self, run_id, y_true, p_lgg):
        """Save full precision-recall curve arrays for later plotting."""
        prec, rec, thrs = precision_recall_curve(np.array(y_true), np.array(p_lgg), pos_label=1)
        path = self.pr_dir / f"{run_id}.npz"
        np.savez_compressed(str(path), precision=prec, recall=rec, thresholds=thrs,
                            y_true=np.array(y_true), p_lgg=np.array(p_lgg))
        print(f"[RunManager] PR curve saved: {path.name}")

    def load_runs(self):   return pd.read_csv(self.runs_csv)   if self.runs_csv.exists()   else pd.DataFrame(columns=self.RUNS_COLS)
    def load_epochs(self, run_id=None):
        df = pd.read_csv(self.epochs_csv) if self.epochs_csv.exists() else pd.DataFrame(columns=self.EPOCHS_COLS)
        return df[df["run_id"] == run_id].copy() if run_id else df
    def load_tests(self):  return pd.read_csv(self.test_csv)  if self.test_csv.exists()   else pd.DataFrame(columns=self.TEST_COLS)

print("[OK] RunManager defined.")

[OK] RunManager defined.


## Model, Loss, Optimizer, Scheduler
Verbatim from `Model 2_parallel_preload.ipynb`. ResNet50 with 4-channel MRI input.

In [9]:
def make_resnet50_inchannels(num_in_channels, num_classes=2, pretrained=True):
    print(f"[Model] ResNet50 | in_ch={num_in_channels} | pretrained={pretrained}")
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    old   = model.conv1
    new   = nn.Conv2d(num_in_channels, old.out_channels, kernel_size=old.kernel_size,
                      stride=old.stride, padding=old.padding, bias=False)
    with torch.no_grad():
        if pretrained:
            w = old.weight
            if num_in_channels <= 3:
                new.weight.copy_(w[:, :num_in_channels, :, :])
            else:
                new.weight[:, :3, :, :].copy_(w)
                mean_w = w.mean(dim=1, keepdim=True)
                for i in range(num_in_channels - 3):
                    new.weight[:, 3+i:4+i, :, :].copy_(mean_w)
        else:
            nn.init.kaiming_normal_(new.weight, mode="fan_out", nonlinearity="relu")
    model.conv1 = new
    model.fc    = nn.Linear(model.fc.in_features, num_classes)
    total = sum(p.numel() for p in model.parameters())
    print(f"[Model] Ready | {total/1e6:.1f}M params")
    return model

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None): super().__init__(); self.gamma=gamma; self.weight=weight
    def forward(self, logits, targets):
        logp=F.log_softmax(logits,dim=1); p=torch.exp(logp)
        pt=p.gather(1,targets.unsqueeze(1)).squeeze(1); logpt=logp.gather(1,targets.unsqueeze(1)).squeeze(1)
        wt=self.weight.gather(0,targets) if self.weight is not None else 1.0
        return (-wt*((1-pt)**self.gamma)*logpt).mean()

def make_loss(train_patients, device, cfg):
    labels = np.array([p.label for p in train_patients])
    n0, n1 = int((labels==0).sum()), int((labels==1).sum())
    if n1 == 0: raise ValueError("No LGG patients.")
    w1 = min(n0/n1, cfg.max_lgg_weight)
    weights = torch.tensor([1.0, w1], dtype=torch.float32, device=device)
    print(f"[Loss] {cfg.loss_type} | HGG w=1.0 | LGG w={w1:.2f}")
    if cfg.loss_type == "weighted_ce": return nn.CrossEntropyLoss(weight=weights)
    if cfg.loss_type == "focal":       return FocalLoss(gamma=cfg.focal_gamma, weight=weights)
    raise ValueError(f"Unknown loss_type: {cfg.loss_type}")

def make_optimizer(model, cfg):
    print(f"[Opt] AdamW | lr={cfg.lr} | wd={cfg.weight_decay}")
    return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

def make_scheduler(optimizer, steps_per_epoch, cfg):
    if not cfg.use_cosine: return None
    total = cfg.epochs * steps_per_epoch; warmup = int(cfg.warmup_epochs * steps_per_epoch)
    print(f"[LR] Cosine+Warmup | total={total} warmup={warmup}")
    def lr_fn(s):
        if s < warmup: return (s+1)/max(1,warmup)
        return 0.5*(1+math.cos(math.pi*(s-warmup)/max(1,total-warmup)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_fn)

print("[OK] Model/loss/optimizer/scheduler defined.")

[OK] Model/loss/optimizer/scheduler defined.


## Enhanced Training Loop

**`train_one_epoch_tracked`** — wraps the DataLoader with a tqdm batch progress bar showing live loss.

**`fit_resnet50_tracked`** — improvements over original:
- Outer epoch loop uses `tqdm` with live `loss / val_AP / val_F1 / best_AP` in postfix
- Records `epoch_time_s` and `lr_current` per epoch
- Calls `run_manager.log_epoch()` after every epoch (if provided)
- Returns `(model, best_thr, best_ap, epoch_history)` for downstream inspection

In [10]:
def train_one_epoch_tracked(model, loader, optimizer, scaler, loss_fn, device,
                              scheduler=None, cfg=None, epoch_num=None, total_epochs=None):
    if cfg is None: cfg = TrainCfg()
    model.train()
    running, n = 0.0, 0
    ep_str = f"Ep {epoch_num}/{total_epochs}" if epoch_num else "Epoch"
    with tqdm(loader, desc=f"  {ep_str}", unit="b", leave=False, position=0) as pbar:
        for batch in pbar:
            x, y, _, __ = batch
            if x.device.type != torch.device(device).type: x = x.to(device, non_blocking=True)
            if y.device.type != torch.device(device).type: y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if cfg.amp and device.startswith("cuda"):
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    logits = model(x); loss = loss_fn(logits, y)
                scaler.scale(loss).backward()
                if cfg.grad_clip > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer); scaler.update()
            else:
                logits = model(x); loss = loss_fn(logits, y)
                loss.backward()
                if cfg.grad_clip > 0: nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                optimizer.step()
            if scheduler is not None: scheduler.step()
            running += float(loss.item()) * x.size(0); n += x.size(0)
            pbar.set_postfix(loss=f"{running/max(1,n):.4f}", refresh=False)
    return running / max(1, n)


def predict_slices(model, loader, device, desc="Predict"):
    model.eval(); y_true, p_lgg, pids = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  {desc}", unit="b", leave=False, position=0):
            x, y, pid, _ = batch
            if x.device.type != torch.device(device).type: x = x.to(device, non_blocking=True)
            probs = torch.softmax(model(x), dim=1)[:, 1]
            y_true.append(y.detach().cpu().numpy())
            p_lgg.append(probs.detach().cpu().numpy())
            pids.extend(list(pid))
    return np.concatenate(y_true), np.concatenate(p_lgg), pids

def aggregate_by_patient(y_true_s, p_lgg_s, pid_s, agg="mean"):
    pd2p, pd2l = {}, {}
    for y, p, pid in zip(y_true_s, p_lgg_s, pid_s):
        pd2p.setdefault(pid, []).append(float(p)); pd2l[pid] = int(y)
    pids = sorted(pd2p)
    probs = [float(np.array(pd2p[p]).mean() if agg=="mean" else np.array(pd2p[p]).max()) for p in pids]
    return np.array([pd2l[p] for p in pids], np.int64), np.array(probs, np.float32), pids

def pick_threshold(y_true, p_lgg, mode="f1", min_recall=0.7):
    prec, rec, thr = precision_recall_curve(y_true, p_lgg, pos_label=1)
    if mode == "f1":
        f1s = [(2*prec[i+1]*rec[i+1]/(prec[i+1]+rec[i+1]+1e-9)) for i in range(len(thr))]
        return float(thr[int(np.argmax(f1s))]) if f1s else 0.5
    best_t, best_p = 0.5, -1.0
    for i in range(len(thr)):
        if rec[i+1] >= min_recall and prec[i+1] > best_p:
            best_p = prec[i+1]; best_t = thr[i]
    return float(best_t)

def compute_report(y_true, p_lgg, thr):
    y_pred = (p_lgg >= thr).astype(np.int64); out = {}
    out["ap_pr_auc(LGG)"] = float(average_precision_score(y_true, p_lgg))
    try:    out["roc_auc"] = float(roc_auc_score(y_true, p_lgg))
    except: out["roc_auc"] = float("nan")
    out["f1(LGG)"]        = float(f1_score(y_true, y_pred, pos_label=1))
    out["balanced_acc"]   = float(balanced_accuracy_score(y_true, y_pred))
    tn, fp, fn, tp        = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    out.update({"tn":float(tn),"fp":float(fp),"fn":float(fn),"tp":float(tp)})
    out["precision(LGG)"] = float(tp/(tp+fp)) if (tp+fp)>0 else 0.0
    out["recall(LGG)"]    = float(tp/(tp+fn)) if (tp+fn)>0 else 0.0
    return out

@torch.no_grad()
def validate_patient_level(model, loader, device, cfg, threshold=None, desc="Eval"):
    y_s, p_s, pid_s = predict_slices(model, loader, device, desc=desc)
    y_p, p_p, _     = aggregate_by_patient(y_s, p_s, pid_s, agg=cfg.patient_agg)
    thr = pick_threshold(y_p, p_p, mode=cfg.tune_threshold_on, min_recall=cfg.min_recall) if threshold is None else float(threshold)
    report = compute_report(y_p, p_p, thr)
    report["threshold"] = float(thr); report["n_patients"] = float(len(y_p))
    return report, thr, y_p, p_p


def fit_resnet50_tracked(train_loader, val_loader, train_patients, cfg, num_in_channels, device,
                          run_manager=None, run_id=None):
    """
    Enhanced fit_resnet50 with tqdm progress bars, per-epoch timing, LR tracking, and RunManager logging.
    Returns: (model, best_thr, best_ap, epoch_history)
    """
    model     = make_resnet50_inchannels(num_in_channels=num_in_channels, num_classes=2, pretrained=True).to(device)
    loss_fn   = make_loss(train_patients, device=device, cfg=cfg)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, steps_per_epoch=len(train_loader), cfg=cfg)
    scaler    = torch.amp.GradScaler("cuda", enabled=(cfg.amp and device.startswith("cuda")))

    best_ap, best_state, best_thr, best_epoch, bad_epochs = -1.0, None, 0.5, 0, 0
    best_metrics  = {}
    epoch_history = []

    val_rep, *_ = validate_patient_level(model, val_loader, device, cfg, desc="Init eval")
    print(f"[Init] Val AP={val_rep['ap_pr_auc(LGG)']:.4f}  F1={val_rep['f1(LGG)']:.4f}  thr={val_rep['threshold']:.3f}")

    print(f"[Train] {cfg.epochs} epochs on {device}")
    # epoch_bar = tqdm(range(1, cfg.epochs + 1), desc="Training", unit="ep", position=0, leave=True)

    # for epoch in epoch_bar:
    for epoch in range(1, cfg.epochs + 1):
        ep_t0   = time.perf_counter()
        tr_loss = train_one_epoch_tracked(
            model, train_loader, optimizer, scaler, loss_fn, device,
            scheduler=scheduler, cfg=cfg, epoch_num=epoch, total_epochs=cfg.epochs,
        )
        ep_time = time.perf_counter() - ep_t0
        lr_now  = optimizer.param_groups[0]["lr"]

        val_rep, val_thr, *_ = validate_patient_level(model, val_loader, device, cfg, desc="Val")
        ap  = val_rep["ap_pr_auc(LGG)"]; f1  = val_rep["f1(LGG)"]
        bal = val_rep["balanced_acc"];   thr = val_rep["threshold"]

        is_best = ap > best_ap + 1e-5
        if is_best:
            best_ap    = ap; best_thr   = thr; best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_metrics = dict(val_rep); bad_epochs = 0
        else:
            bad_epochs += 1

        # epoch_bar.set_postfix(
        #     loss=f"{tr_loss:.4f}", val_AP=f"{ap:.4f}",
        #     val_F1=f"{f1:.4f}",  best_AP=f"{best_ap:.4f}", refresh=True
        # )
        star = " ★ new best" if is_best else ""
        tqdm.write(f"  [Ep {epoch:02d}/{cfg.epochs}] loss={tr_loss:.4f} | AP={ap:.4f} F1={f1:.4f} "
                   f"Bal={bal:.4f} Prec={val_rep['precision(LGG)']:.4f} Rec={val_rep['recall(LGG)']:.4f} "
                   f"thr={thr:.3f} lr={lr_now:.2e} t={ep_time:.0f}s{star}")

        ep_rec = {"epoch": epoch, "train_loss": tr_loss, "epoch_time_s": ep_time,
                  "lr_current": lr_now, "is_best": is_best, **{f"val_{k}": v for k, v in val_rep.items()}}
        epoch_history.append(ep_rec)

        if run_manager is not None and run_id is not None:
            run_manager.log_epoch(run_id, epoch, tr_loss, ep_time, lr_now, val_rep, is_best)

        if bad_epochs >= cfg.early_stop_patience:
            print(f"  [EarlyStop] No AP improvement for {cfg.early_stop_patience} epochs.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[Train] Restored best model | epoch={best_epoch} | AP={best_ap:.4f} | thr={best_thr:.3f}")

    return model, best_thr, best_ap, epoch_history, best_epoch, best_metrics

print("[OK] Training loop and evaluation functions defined.")

[OK] Training loop and evaluation functions defined.


## Data Setup
Run **once per session**. Patient scan and split are reused across all runs in this session.

In [13]:
# ── Paths ─────────────────────────────────────────────────────────────────────
if os.getenv('CUSTOM_COLAB_SERVER'):
    BASE_DIR = r"/content/Dataset/PKG - UCSF-PDGM Version 5"
else:
    BASE_DIR = r"D:\Colab\Dataset\PKG - UCSF-PDGM Version 5"

NIFTI_ROOT = os.path.join(BASE_DIR, "UCSF-PDGM-v5")
META_CSV   = os.path.join(BASE_DIR, "UCSF-PDGM-metadata_v5.csv")

# Reports live next to this notebook
if os.getenv('CUSTOM_COLAB_SERVER'):
    NOTEBOOK_DIR = Path('/content')
else:
    NOTEBOOK_DIR = Path().resolve()
REPORTS_DIR  = NOTEBOOK_DIR / "Reports"

print(f"[Paths] BASE_DIR   : {BASE_DIR}")
print(f"[Paths] Reports    : {REPORTS_DIR}")
print(f"[Paths] Base exists: {os.path.isdir(BASE_DIR)}")

# ── Labels + scan ─────────────────────────────────────────────────────────────
labels_by_pid, df_meta = build_labels_from_pdgm_v5(META_CSV)
patients = scan_ucsf_pdgm_v5(NIFTI_ROOT, labels_by_pid)
train_patients, val_patients, test_patients = stratified_patient_split(patients)

y_all = np.array([p.label for p in patients])
print(f"\n[Dataset] Total={len(patients)} HGG={int((y_all==0).sum())} LGG={int((y_all==1).sum())}")

[Paths] BASE_DIR   : /content/Dataset/PKG - UCSF-PDGM Version 5
[Paths] Reports    : /content/Reports
[Paths] Base exists: True
[Labels] 495 | LGG(1)=56 | HGG(0)=439
[Scan] /content/Dataset/PKG - UCSF-PDGM Version 5/UCSF-PDGM-v5
[Scan] Found 495 | Skipped 0
[Split] Train:{'N': 345, 'LGG': np.int64(39), 'HGG': np.int64(306)} Val:{'N': 75, 'LGG': np.int64(9), 'HGG': np.int64(66)} Test:{'N': 75, 'LGG': np.int64(8), 'HGG': np.int64(67)}

[Dataset] Total=495 HGG=439 LGG=56


In [12]:
# ── RunManager (init once per session) ────────────────────────────────────────
run_manager = RunManager(REPORTS_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")
print(f"[Runs]   {len(run_manager.load_runs())} runs logged so far")

[RunManager] Ready → /content/Reports
[Device] cuda
[Runs]   60 runs logged so far


---
## Start a New Run

**Edit cells 24–26 for each new experiment.** Cell 25 (build) and 26 (train+test) are self-contained
— just run them top-to-bottom after editing the config.

### Quick checklist before each run
- [ ] Updated `RunCfg.note` to describe what you're testing
- [ ] Updated `RunCfg.tag` for a readable run_id suffix
- [ ] Changed the hyperparameter(s) you want to evaluate
- [ ] *(Optional)* Changed `RunCfg.seed` for an independent replicate

In [14]:
# ── Configure this run (edit me!) ─────────────────────────────────────────────
run_cfg = RunCfg(
    note = "top32k focal",
    tag  = "top32k",
    seed = 42,
)

train_cfg = TrainCfg(
    epochs=20, lr=3e-3, weight_decay=1e-4,
    loss_type="focal",
    amp=True, grad_clip=1.0,
    use_cosine=True, warmup_epochs=1,
    early_stop_patience=5,
)

cfg = PreprocCfg(
    topk=32, context_slices=2, bbox_margin=24,
    out_hw=(224, 224),
    add_tumor_mask_channel=True, norm_mode="robust_z",
)

parallel_cfg = ParallelCfg(
    n_workers=-1, device=DEVICE,
)

# ── Seed everything ───────────────────────────────────────────────────────────
random.seed(run_cfg.seed)
np.random.seed(run_cfg.seed)
torch.manual_seed(run_cfg.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(run_cfg.seed)

NUM_IN_CH = len(cfg.mods) + (1 if cfg.add_tumor_mask_channel else 0)
print(f"[Config] Note: {run_cfg.note!r}")
print(f"[Config] TrainCfg: lr={train_cfg.lr}, loss={train_cfg.loss_type}, "
      f"epochs={train_cfg.epochs}, seed={run_cfg.seed}")
print(f"[Config] in_channels={NUM_IN_CH}")

[Config] Note: 'top32k focal'
[Config] TrainCfg: lr=0.003, loss=focal, epochs=20, seed=42
[Config] in_channels=5


In [14]:
# ── Build datasets + DataLoaders ──────────────────────────────────────────────
print("[Build] Starting parallel dataset preparation ...")
build_t0 = time.perf_counter()

train_ds = ParallelDatasetBuilder.build(train_patients, cfg, parallel_cfg, "train")
val_ds   = ParallelDatasetBuilder.build(val_patients,   cfg, parallel_cfg, "val")
test_ds  = ParallelDatasetBuilder.build(test_patients,  cfg, parallel_cfg, "test")

print(f"[Build] All splits ready in {time.perf_counter()-build_t0:.1f}s")

train_sampler = PreloadedPatientBalancedSampler(train_ds, batch_patients_per_class=4, seed=run_cfg.seed)
train_loader  = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=0, pin_memory=False)
val_loader    = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0, pin_memory=False)
test_loader   = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=0, pin_memory=False)

print(f"[Loaders] train={len(train_loader)} batches | val={len(val_loader)} | test={len(test_loader)}")

# Sanity check
xb, yb, _, __ = next(iter(train_loader))
print(f"[Sanity] x={tuple(xb.shape)} {xb.dtype} on {xb.device} | y={tuple(yb.shape)}")

[Build] Starting parallel dataset preparation ...

[Phase 1/3 | train] Estimate: 345 patients, ~12420 slices, ~12.46 GB
  [WARN] Exceeds max_ram_estimate_gb=12.0. Consider reducing topk or using device='cpu'.
[Phase 2/3 | train] Processing (8 threads) ...


  [train]: 100%|██████████| 345/345 [01:17<00:00,  4.45pt/s, fail=0, pid=DGM-0011, slices=12732]


  Done: 345/345 OK | 0 failed | 12732 slices | 81.0s
[Phase 3/3 | train] Stacking 12732 slices ...
  Shape: (12732, 5, 224, 224) | 12.78 GB
  Stack+transfer: 5.5s | Total: 86.5s
  PreloadedSliceDataset | 12732 slices | 345 patients | X=(12732, 5, 224, 224) torch.float32 on cuda:0 | {0: 11281, 1: 1451}

[Phase 1/3 | val] Estimate: 75 patients, ~2700 slices, ~2.71 GB
[Phase 2/3 | val] Processing (8 threads) ...


  [val]: 100%|██████████| 75/75 [00:15<00:00,  4.93pt/s, fail=0, pid=DGM-0295, slices=2773]


  Done: 75/75 OK | 0 failed | 2773 slices | 16.5s
[Phase 3/3 | val] Stacking 2773 slices ...
  Shape: (2773, 5, 224, 224) | 2.78 GB
  Stack+transfer: 1.3s | Total: 17.8s
  PreloadedSliceDataset | 2773 slices | 75 patients | X=(2773, 5, 224, 224) torch.float32 on cuda:0 | {0: 2442, 1: 331}

[Phase 1/3 | test] Estimate: 75 patients, ~2700 slices, ~2.71 GB
[Phase 2/3 | test] Processing (8 threads) ...


  [test]: 100%|██████████| 75/75 [00:15<00:00,  4.70pt/s, fail=0, pid=DGM-0264, slices=2773]


  Done: 75/75 OK | 0 failed | 2773 slices | 16.0s
[Phase 3/3 | test] Stacking 2773 slices ...
  Shape: (2773, 5, 224, 224) | 2.78 GB
  Stack+transfer: 1.4s | Total: 17.5s
  PreloadedSliceDataset | 2773 slices | 75 patients | X=(2773, 5, 224, 224) torch.float32 on cuda:0 | {0: 2481, 1: 292}
[Build] All splits ready in 121.9s
[Sampler] HGG=306 LGG=39 k=4 ~76 batches/epoch
[Loaders] train=76 batches | val=174 | test=174
[Sanity] x=(8, 5, 224, 224) torch.float32 on cuda:0 | y=(8,)


In [15]:
# ── Train + evaluate + save ───────────────────────────────────────────────────
run_id = run_manager.new_run_id(run_cfg.tag)
print(f"\n{'='*65}")
print(f"[Run] ID    : {run_id}")
print(f"[Run] Note  : {run_cfg.note!r}")
print(f"{'='*65}")

# Log run start (config + dataset counts)
run_manager.start_run(
    run_id, run_cfg, cfg, train_cfg,
    split_counts  = {"train": len(train_patients), "val": len(val_patients), "test": len(test_patients)},
    slice_counts  = {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
    class_counts  = {"hgg": sum(1 for p in train_patients if p.label==0),
                     "lgg": sum(1 for p in train_patients if p.label==1)},
    n_workers     = parallel_cfg.resolved_n_workers(),
    device        = DEVICE,
    all_patients  = patients,
)

# ── Train ─────────────────────────────────────────────────────────────────────
run_t0 = time.perf_counter()
model, best_thr, best_ap, history, best_epoch, best_metrics = fit_resnet50_tracked(
    train_loader, val_loader, train_patients, train_cfg,
    num_in_channels=NUM_IN_CH, device=DEVICE,
    run_manager=run_manager, run_id=run_id,
)
duration = time.perf_counter() - run_t0

# ── Save best model ───────────────────────────────────────────────────────────
model_path = run_manager.models_dir / f"{run_id}.pt"
torch.save(model.state_dict(), str(model_path))
print(f"[Save] Best model → {model_path.name}")

# ── Finish run (update runs.csv) ──────────────────────────────────────────────
run_manager.finish_run(
    run_id,
    epochs_actual = len(history),
    best_epoch    = best_epoch,
    best_metrics  = best_metrics,
    model_path    = model_path,
    duration_s    = duration,
)

# ── Test evaluation ───────────────────────────────────────────────────────────
print("\n[Test] Evaluating on test set ...")
test_report, _, y_tp, p_tp = validate_patient_level(model, test_loader, DEVICE, train_cfg, threshold=best_thr, desc="Test eval")
run_manager.log_test(run_id, test_report)

# ── Save PR curve (reuses predictions from test eval above) ────────────────
run_manager.log_pr_curve(run_id, y_tp, p_tp)

print(f"\n{'='*65}")
print(f"[Done] {run_id}")
print(f"  Best Val AP : {best_ap:.4f}  (epoch {best_epoch})")
print(f"  Test AP     : {test_report['ap_pr_auc(LGG)']:.4f}")
print(f"  Test F1     : {test_report['f1(LGG)']:.4f}")
print(f"  Test BalAcc : {test_report['balanced_acc']:.4f}")
print(f"  Duration    : {duration:.0f}s")
print(f"{'='*65}")


[Run] ID    : run_20260324_205123_top32k
[Run] Note  : 'top32k focal'
[RunManager] Run started: run_20260324_205123_top32k
[Model] ResNet50 | in_ch=5 | pretrained=True
[Model] Ready | 23.5M params
[Loss] focal | HGG w=1.0 | LGG w=7.85
[Opt] AdamW | lr=0.003 | wd=0.0001
[LR] Cosine+Warmup | total=1520 warmup=76


[Init] Val AP=0.1323  F1=0.3019  thr=0.000
[Train] 20 epochs on cuda


  [Ep 01/20] loss=0.6894 | AP=0.4239 F1=0.5714 Bal=0.7879 Prec=0.5000 Rec=0.6667 thr=0.991 lr=3.00e-03 t=41s ★ new best


  [Ep 02/20] loss=0.4806 | AP=0.1647 F1=0.3103 Bal=0.6970 Prec=0.1837 Rec=1.0000 thr=0.864 lr=2.98e-03 t=26s


  [Ep 03/20] loss=0.5559 | AP=0.5552 F1=0.7000 Bal=0.8586 Prec=0.6364 Rec=0.7778 thr=0.816 lr=2.92e-03 t=26s ★ new best


...

  [Ep 13/20] loss=0.1351 | AP=0.8409 F1=0.8000 Bal=0.9217 Prec=0.7273 Rec=0.8889 thr=0.711 lr=8.97e-04 t=34s


  [Ep 14/20] loss=0.1801 | AP=0.8156 F1=0.7619 Bal=0.9141 Prec=0.6667 Rec=0.8889 thr=0.629 lr=6.80e-04 t=34s


  [Ep 15/20] loss=0.1152 | AP=0.7843 F1=0.8000 Bal=0.9217 Prec=0.7273 Rec=0.8889 thr=0.735 lr=4.84e-04 t=34s


  [Ep 16/20] loss=0.1306 | AP=0.7651 F1=0.7619 Bal=0.9141 Prec=0.6667 Rec=0.8889 thr=0.586 lr=3.16e-04 t=34s
  [EarlyStop] No AP improvement for 5 epochs.
[Train] Restored best model | epoch=11 | AP=0.8969 | thr=0.682
[Save] Best model → run_20260324_205123_top32k.pt
[RunManager] Run finished: run_20260324_205123_top32k | 2575s | best_val_ap=0.8969

[Test] Evaluating on test set ...


[RunManager] Test logged for run_20260324_205123_top32k
[RunManager] PR curve saved: run_20260324_205123_top32k.npz

[Done] run_20260324_205123_top32k
  Best Val AP : 0.8969  (epoch 11)
  Test AP     : 0.8370
  Test F1     : 0.7778
  Test BalAcc : 0.9151
  Duration    : 2575s


In [16]:
# ── Last run summary (read-only convenience cell) ─────────────────────────────
# Use this at the START of a new session to see what your last run looked like,
# so you can copy-modify the config for your next experiment.
try:
    df_runs = run_manager.load_runs()
    if df_runs.empty:
        print("No runs logged yet.")
    else:
        last = df_runs.iloc[-1]
        cfg_cols = ["run_id","run_note","tag","seed",
                    "lr","weight_decay","loss_type","focal_gamma",
                    "epochs_cfg","epochs_actual","best_epoch",
                    "topk","context_slices","norm_mode",
                    "best_val_ap","best_val_f1","best_val_balanced_acc","best_val_threshold"]
        print("[Last Run]")
        for c in cfg_cols:
            if c in last.index:
                print(f"  {c:35s} = {last[c]}")
        print(f"\nTotal runs logged: {len(df_runs)}")
except Exception as e:
    print(f"Could not read runs: {e}")

[Last Run]
  run_id                              = run_20260324_205123_top32k
  run_note                            = top32k focal
  tag                                 = top32k
  seed                                = 42
  lr                                  = 0.003
  weight_decay                        = 0.0001
  loss_type                           = focal
  focal_gamma                         = 2.0
  epochs_cfg                          = 20
  epochs_actual                       = 16.0
  best_epoch                          = 11.0
  topk                                = 32
  context_slices                      = 2
  norm_mode                           = robust_z
  best_val_ap                         = 0.8969496552829888
  best_val_f1                         = 0.8571428571428571
  best_val_balanced_acc               = 0.9772727272727272
  best_val_threshold                  = 0.681505024433136

Total runs logged: 10


In [15]:
loop_cfgs: List[Tuple[RunCfg, PreprocCfg, TrainCfg, bool]] = [
    (RunCfg(note='topk16 robust_z 1e-02',       tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    (RunCfg(note='topk16 robust_z 1e-04',       tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z 3e-02',       tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z 3e-04',       tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z 5e-02',       tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z 5e-04',       tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z tumor 1e-02', tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    (RunCfg(note='topk16 robust_z tumor 1e-04', tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z tumor 3e-02', tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z tumor 3e-04', tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z tumor 5e-02', tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk16 robust_z tumor 5e-04', tag=''), PreprocCfg(topk=16, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 1e-02',          tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    # (RunCfg(note='topk16 pct01 1e-04',          tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 3e-02',          tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 3e-04',          tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 5e-02',          tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 5e-04',          tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 tumor 1e-02',    tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    # (RunCfg(note='topk16 pct01 tumor 1e-04',    tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 tumor 3e-02',    tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 tumor 3e-04',    tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 tumor 5e-02',    tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk16 pct01 tumor 5e-04',    tag=''), PreprocCfg(topk=16, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z 1e-02',       tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    (RunCfg(note='topk24 robust_z 1e-04',       tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z 3e-02',       tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z 3e-04',       tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z 5e-02',       tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z 5e-04',       tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z tumor 1e-02', tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    (RunCfg(note='topk24 robust_z tumor 1e-04', tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z tumor 3e-02', tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z tumor 3e-04', tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z tumor 5e-02', tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk24 robust_z tumor 5e-04', tag=''), PreprocCfg(topk=24, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 1e-02',          tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    # (RunCfg(note='topk24 pct01 1e-04',          tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 3e-02',          tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 3e-04',          tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 5e-02',          tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 5e-04',          tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 tumor 1e-02',    tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    # (RunCfg(note='topk24 pct01 tumor 1e-04',    tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 tumor 3e-02',    tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 tumor 3e-04',    tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 tumor 5e-02',    tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk24 pct01 tumor 5e-04',    tag=''), PreprocCfg(topk=24, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk32 robust_z 1e-02',       tag=''), PreprocCfg(topk=32, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    (RunCfg(note='topk32 robust_z 1e-04',       tag=''), PreprocCfg(topk=32, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk32 robust_z 3e-02',       tag=''), PreprocCfg(topk=32, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk32 robust_z 3e-04',       tag=''), PreprocCfg(topk=32, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk32 robust_z 5e-02',       tag=''), PreprocCfg(topk=32, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    (RunCfg(note='topk32 robust_z 5e-04',       tag=''), PreprocCfg(topk=32, norm_mode='robust_z', out_hw=(224,224), add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 robust_z tumor 1e-02', tag=''), PreprocCfg(topk=32, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    # (RunCfg(note='topk32 robust_z tumor 1e-04', tag=''), PreprocCfg(topk=32, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 robust_z tumor 3e-02', tag=''), PreprocCfg(topk=32, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 robust_z tumor 3e-04', tag=''), PreprocCfg(topk=32, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 robust_z tumor 5e-02', tag=''), PreprocCfg(topk=32, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 robust_z tumor 5e-04', tag=''), PreprocCfg(topk=32, norm_mode='robust_z', add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 1e-02',          tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    # (RunCfg(note='topk32 pct01 1e-04',          tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 3e-02',          tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 3e-04',          tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 5e-02',          tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 5e-04',          tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 tumor 1e-02',    tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-02, weight_decay=1e-04, early_stop_patience=10), True),
    # (RunCfg(note='topk32 pct01 tumor 1e-04',    tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=1e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 tumor 3e-02',    tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 tumor 3e-04',    tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=3e-04, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 tumor 5e-02',    tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-02, weight_decay=1e-04, early_stop_patience=10), False),
    # (RunCfg(note='topk32 pct01 tumor 5e-04',    tag=''), PreprocCfg(topk=32, norm_mode='pct01',    add_tumor_mask_channel=True),  TrainCfg(epochs=20, lr=5e-04, weight_decay=1e-04, early_stop_patience=10), False),
]

print(f"[OK] {len(loop_cfgs)} configs ready for training.")

[OK] 30 configs ready for training.


In [16]:
for run_cfg, cfg, train_cfg, reprep in loop_cfgs:
    print(f"[LOG] Training with config {run_cfg}, {cfg}, {train_cfg}")
    # ── Seed everything ───────────────────────────────────────────────────────────
    random.seed(run_cfg.seed)
    np.random.seed(run_cfg.seed)
    torch.manual_seed(run_cfg.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(run_cfg.seed)
    
    NUM_IN_CH = len(cfg.mods) + (1 if cfg.add_tumor_mask_channel else 0)
    print(f"[Config] Note: {run_cfg.note!r}")
    print(f"[Config] TrainCfg: lr={train_cfg.lr}, loss={train_cfg.loss_type}, "
          f"epochs={train_cfg.epochs}, seed={run_cfg.seed}")
    print(f"[Config] in_channels={NUM_IN_CH}")
    if reprep:
        # ── Build datasets + DataLoaders ──────────────────────────────────────────────
        print("[Build] Starting parallel dataset preparation ...")
        build_t0 = time.perf_counter()

        train_ds = ParallelDatasetBuilder.build(train_patients, cfg, parallel_cfg, "train")
        val_ds   = ParallelDatasetBuilder.build(val_patients,   cfg, parallel_cfg, "val")
        test_ds  = ParallelDatasetBuilder.build(test_patients,  cfg, parallel_cfg, "test")

        print(f"[Build] All splits ready in {time.perf_counter()-build_t0:.1f}s")

        train_sampler = PreloadedPatientBalancedSampler(train_ds, batch_patients_per_class=4, seed=run_cfg.seed)
        train_loader  = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=0, pin_memory=False)
        val_loader    = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0, pin_memory=False)
        test_loader   = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=0, pin_memory=False)

        print(f"[Loaders] train={len(train_loader)} batches | val={len(val_loader)} | test={len(test_loader)}")

        # Sanity check
        xb, yb, _, __ = next(iter(train_loader))
        print(f"[Sanity] x={tuple(xb.shape)} {xb.dtype} on {xb.device} | y={tuple(yb.shape)}")
    # ── Train + evaluate + save ───────────────────────────────────────────────────
    run_id = run_manager.new_run_id(run_cfg.tag)
    print(f"\n{'='*65}")
    print(f"[Run] ID    : {run_id}")
    print(f"[Run] Note  : {run_cfg.note!r}")
    print(f"{'='*65}")
    
    # Log run start (config + dataset counts)
    run_manager.start_run(
        run_id, run_cfg, cfg, train_cfg,
        split_counts  = {"train": len(train_patients), "val": len(val_patients), "test": len(test_patients)},
        slice_counts  = {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)},
        class_counts  = {"hgg": sum(1 for p in train_patients if p.label==0),
                         "lgg": sum(1 for p in train_patients if p.label==1)},
        n_workers     = parallel_cfg.resolved_n_workers(),
        device        = DEVICE,
        all_patients  = patients,
    )
    
    # ── Train ─────────────────────────────────────────────────────────────────────
    run_t0 = time.perf_counter()
    model, best_thr, best_ap, history, best_epoch, best_metrics = fit_resnet50_tracked(
        train_loader, val_loader, train_patients, train_cfg,
        num_in_channels=NUM_IN_CH, device=DEVICE,
        run_manager=run_manager, run_id=run_id,
    )
    duration = time.perf_counter() - run_t0
    
    # ── Save best model ───────────────────────────────────────────────────────────
    model_path = run_manager.models_dir / f"{run_id}.pt"
    torch.save(model.state_dict(), str(model_path))
    print(f"[Save] Best model → {model_path.name}")
    
    # ── Finish run (update runs.csv) ──────────────────────────────────────────────
    run_manager.finish_run(
        run_id,
        epochs_actual = len(history),
        best_epoch    = best_epoch,
        best_metrics  = best_metrics,
        model_path    = model_path,
        duration_s    = duration,
    )
    
    # ── Test evaluation ───────────────────────────────────────────────────────────
    print("\n[Test] Evaluating on test set ...")
    test_report, _, y_tp, p_tp = validate_patient_level(model, test_loader, DEVICE, train_cfg, threshold=best_thr, desc="Test eval")
    run_manager.log_test(run_id, test_report)
    
    # ── Save PR curve (reuses predictions from test eval above) ────────────────
    run_manager.log_pr_curve(run_id, y_tp, p_tp)
    
    print(f"\n{'='*65}")
    print(f"[Done] {run_id}")
    print(f"  Best Val AP : {best_ap:.4f}  (epoch {best_epoch})")
    print(f"  Test AP     : {test_report['ap_pr_auc(LGG)']:.4f}")
    print(f"  Test F1     : {test_report['f1(LGG)']:.4f}")
    print(f"  Test BalAcc : {test_report['balanced_acc']:.4f}")
    print(f"  Duration    : {duration:.0f}s")
    print(f"{'='*65}")

[LOG] Training with config RunCfg(note='topk16 robust_z 1e-02', tag='', seed=42), PreprocCfg(mods=('T1', 'T1c', 'T2', 'FLAIR'), out_hw=(256, 256), topk=16, context_slices=2, include_context_ratio=0.25, bbox_margin=24, min_bbox_size=32, norm_mode='robust_z', clip_z=5.0, eps=1e-06, add_tumor_mask_channel=False), TrainCfg(epochs=20, lr=0.01, weight_decay=0.0001, amp=True, grad_clip=1.0, loss_type='weighted_ce', focal_gamma=2.0, max_lgg_weight=8.0, use_cosine=True, warmup_epochs=1, early_stop_patience=10, tune_threshold_on='f1', min_recall=0.7, patient_agg='mean')
[Config] Note: 'topk16 robust_z 1e-02'
[Config] TrainCfg: lr=0.01, loss=weighted_ce, epochs=20, seed=42
[Config] in_channels=4
[Build] Starting parallel dataset preparation ...

[Phase 1/3 | train] Estimate: 345 patients, ~6900 slices, ~7.24 GB
[Phase 2/3 | train] Processing (8 threads) ...


  [train]: 100%|██████████| 345/345 [01:08<00:00,  5.07pt/s, fail=0, pid=DGM-0011, slices=7226]


  Done: 345/345 OK | 0 failed | 7226 slices | 73.2s
[Phase 3/3 | train] Stacking 7226 slices ...
  Shape: (7226, 4, 256, 256) | 7.58 GB
  Stack+transfer: 4.5s | Total: 77.8s
  PreloadedSliceDataset | 7226 slices | 345 patients | X=(7226, 4, 256, 256) torch.float32 on cuda:0 | {0: 6423, 1: 803}

[Phase 1/3 | val] Estimate: 75 patients, ~1500 slices, ~1.57 GB
[Phase 2/3 | val] Processing (8 threads) ...


  [val]: 100%|██████████| 75/75 [00:16<00:00,  4.60pt/s, fail=0, pid=DGM-0295, slices=1545]


  Done: 75/75 OK | 0 failed | 1545 slices | 16.3s
[Phase 3/3 | val] Stacking 1545 slices ...
  Shape: (1545, 4, 256, 256) | 1.62 GB
  Stack+transfer: 0.7s | Total: 17.1s
  PreloadedSliceDataset | 1545 slices | 75 patients | X=(1545, 4, 256, 256) torch.float32 on cuda:0 | {0: 1362, 1: 183}

[Phase 1/3 | test] Estimate: 75 patients, ~1500 slices, ~1.57 GB
[Phase 2/3 | test] Processing (8 threads) ...


...

  [Ep 08/20] loss=0.0496 | AP=0.6021 F1=0.6000 Bal=0.7955 Prec=0.5455 Rec=0.6667 thr=0.516 lr=3.50e-04 t=2s


  [Ep 09/20] loss=0.0782 | AP=0.5629 F1=0.5600 Bal=0.8207 Prec=0.4375 Rec=0.7778 thr=0.315 lr=3.11e-04 t=2s


  [Ep 10/20] loss=0.0460 | AP=0.5159 F1=0.5714 Bal=0.7879 Prec=0.5000 Rec=0.6667 thr=0.435 lr=2.71e-04 t=2s


  [Ep 11/20] loss=0.0371 | AP=0.5486 F1=0.5714 Bal=0.7879 Prec=0.5000 Rec=0.6667 thr=0.396 lr=2.29e-04 t=2s
  [EarlyStop] No AP improvement for 10 epochs.
[Train] Restored best model | epoch=1 | AP=0.8409 | thr=0.582
[Save] Best model → run_20260328_124421.pt
[RunManager] Run finished: run_20260328_124421 | 44s | best_val_ap=0.8409

[Test] Evaluating on test set ...


[RunManager] Test logged for run_20260328_124421
[RunManager] PR curve saved: run_20260328_124421.npz

[Done] run_20260328_124421
  Best Val AP : 0.8409  (epoch 1)
  Test AP     : 0.8639
  Test F1     : 0.5833
  Test BalAcc : 0.8703
  Duration    : 44s
